In [1]:
!pip install BeautifulSoup4

# Task
Generate a Python script to fetch data from the URL "https://www.encuentra24.com/el-salvador-es/bienes-raices/map/data?q=lat.13.111763279623297|lng.-88.81290103085456|zoom.8&sort=f_added&dir=desc&page=1&list=categoryregionmap&ulat=11.334823499024711&ulng=-89.99393130429208&llat=14.875959926576407&llng=-87.63187075741708", paginate through all available pages, and write the collected data to a Google Sheet.

## Fetch initial data

### Subtask:
Make an initial request to the API to get the first page of data and the total number of results.


**Reasoning**:
Import the `requests` library, define the base URL, construct the URL for the first page, make a GET request, parse the JSON response, and extract the data and total number of results.



In [6]:
import requests

base_url = "https://www.encuentra24.com"
params = {
    "q": "lat.14.04352777514262|lng.-88.94262584401595|zoom.9",
    "sort": "f_added",
    "dir": "desc",
    "page": 1,
    "list": "categorymap",
    "ulat": 13.11578367262148,
    "ulng": -90.2747181291722,
    "llat": 14.96752953517505,
    "llng": -87.6105335588597
}

response = requests.get(f"{base_url}/el-salvador-es/bienes-raices/map/data?", params=params)
response.raise_for_status() # Raise an exception for bad status codes
data = response.json()

first_page_data = data.get("data").get("count",0)
total_results =  data.get("data").get("total",0)

In [12]:
from bs4 import BeautifulSoup
import time # Import the time module for adding delays

all_listing_details = [] # List to store details for all listings

# Iterate through each listing in the 'data'
for item in data.get("data").get("data", []):
    listing_url = base_url + item.get("lin", "") # Construct the full URL for the listing
    if not listing_url:
        continue # Skip if 'lin' field is empty

    try:
        listing_response = requests.get(listing_url)
        listing_response.raise_for_status() # Raise an exception for bad status codes
        listing_soup = BeautifulSoup(listing_response.content, 'html.parser')

        # Extract details from the listing page
        # These selectors are based on the provided example URL, and may need adjustment
        place_element = listing_soup.select_one("span.text-muted.d-inline-block.location")
        place = place_element.text.strip() if place_element else "N/A"

        date_published_element = listing_soup.select_one("span.pl-3")
        date_published = date_published_element.text.strip() if date_published_element else "N/A"

        # Extracting details from the table-like structure
        details_dict = {}
        detail_elements = listing_soup.select("ul.list-group.list-group-flush li.list-group-item")
        for detail in detail_elements:
            key_element = detail.select_one("b")
            value_element = detail.select_one("span")
            if key_element and value_element:
                key = key_element.text.strip().replace(":", "")
                value = value_element.text.strip()
                details_dict[key] = value

        bathrooms = details_dict.get("bth", "N/A")
        garages = details_dict.get("prk", "N/A")
        rooms = details_dict.get("rom", "N/A")
        price = details_dict.get("pri", "N/A")


        all_listing_details.append({
            "lid": item.get("lid", "N/A"),
            "title": item.get("tit", "N/A"),
            "date": item.get("dat", "N/A"),
            "latitude": item.get("lat", "N/A"),
            "longitude": item.get("lng", "N/A"),
            "image_url": item.get("img", "N/A"),
            "listing_url": listing_url,
            "place": place,
            "date_published": date_published,
            "bathrooms": bathrooms,
            "garages": garages,
            "rooms": rooms,
            "price": price
        })

    except requests.exceptions.RequestException as e:
        print(f"Error fetching listing details for {listing_url}: {e}")
    except Exception as e:
        print(f"Error parsing listing details for {listing_url}: {e}")
    print(all_listing_details[len(all_listing_details)-1])
    print(f"Processed {len(all_listing_details)} out of {total_results} listings.")
 #   time.sleep(1) # Add a small delay between requests to avoid overwhelming the server

# Now, all_listing_details contains the scraped information for the first page.
# You would need to implement pagination to get data from all pages.

{'lid': '28291168', 'title': 'CASA EN CUIDAD ARCE CERCA DEL CENTRO', 'date': '20/09/2025', 'latitude': 13.833163, 'longitude': -89.44356, 'image_url': 'https://photos.encuentra24.com/t_or_cvr_s/f_auto/v1/sv/28/29/11/68/28291168_7010fe', 'listing_url': 'https://www.encuentra24.com/el-salvador-es/bienes-raices-venta-de-propiedades-casas/casa-en-cuidad-arce-cerca-del-centro/28291168', 'place': 'N/A', 'date_published': 'N/A', 'bathrooms': 'N/A', 'garages': 'N/A', 'rooms': 'N/A', 'price': 'N/A'}
Processed 1 out of 2370 listings.
{'lid': '31025949', 'title': 'Casa en venta, estilo hacienda en Residencial ...', 'date': '20/09/2025', 'latitude': 13.569877, 'longitude': -89.26166, 'image_url': 'https://photos.encuentra24.com/t_or_cvr_s/f_auto/v1/sv/31/02/59/49/31025949_d4216b', 'listing_url': 'https://www.encuentra24.com/el-salvador-es/bienes-raices-venta-de-propiedades-casas/casa-en-venta-estilo-hacienda-en-residencial-la-hacienda/31025949', 'place': 'N/A', 'date_published': 'N/A', 'bathrooms'

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os

# Ensure Google Drive is mounted
try:
    drive_root = '/content/drive/My Drive/'
    os.makedirs(drive_root, exist_ok=True)
except Exception as e:
    print(f"Error accessing Google Drive: {e}")
    print("Please ensure Google Drive is mounted by running the previous cell.")

# Convert the data to a pandas DataFrame
df = pd.DataFrame(all_data)

# Define the file path in Google Drive
csv_file_path = os.path.join(drive_root, 'encuentra24_v2.csv')

# Save the DataFrame to a CSV file
df.to_csv(csv_file_path, index=False)

print(f"Data successfully saved to {csv_file_path}")

Data successfully saved to /content/drive/My Drive/encuentra24.csv


In [ ]:
df.head()

,lid,tit,dat,lat,lng,img,fea,feg,fep,fes,...,rom,prk,bth,des,lin,exct,Localización,Publicado,Precio/M² de terreno,map
0,31143629,Lotes residenciales Hacienda los naranjos .,29/08/2025,13.988708,-89.552310,https://photos.encuentra24.com/t_or_cvr_s/f_au...,False,False,True,False,...,,,,¡Anteproyecto Preventa – Terrenos en Venta en ...,/el-salvador-es/bienes-raices-venta-de-propied...,True,Santa Ana,29/08/2025,$100.00,"https://www.google.com/maps/search/13.9887085,..."
1,31177313,Casa a estrenar -Residencial Valterra - ...,29/08/2025,13.784378,-89.193275,https://photos.encuentra24.com/t_or_cvr_s/f_au...,False,False,True,False,...,3,2,2.5,"105 metros cuadrados de construcción, 2 nivele...",/el-salvador-es/bienes-raices-venta-de-propied...,True,Apopa,29/08/2025,"$1,456","https://www.google.com/maps/search/13.784378,+..."
2,31140486,Espectacular Terreno de playa en La Costa del ...,28/08/2025,13.306776,-88.876650,https://photos.encuentra24.com/t_or_cvr_s/f_au...,False,False,True,False,...,,,,Espectacular Terreno de playa en venta en La P...,/el-salvador-es/bienes-raices-venta-de-propied...,True,La Paz,28/08/2025,$160.41,"https://www.google.com/maps/search/13.306776,+..."
3,31248008,Venta de casa en Residencial Miramar San ...,29/08/2025,13.586751,-89.281450,https://photos.encuentra24.com/t_or_cvr_s/f_au...,False,False,True,False,...,3,2,3.5,Venta de casa en Residencial Miramar San José ...,/el-salvador-es/bienes-raices-venta-de-propied...,True,San José Villanueva,29/08/2025,"$1,866","https://www.google.com/maps/search/13.586751,+..."
4,31181089,EN VENTA Casa de lujo en Cumbres de ...,28/08/2025,13.671771,-89.235954,https://photos.encuentra24.com/t_or_cvr_s/f_au...,False,False,True,False,...,4,6,4,EN VENTA Casa de lujo en Cumbres de Cuscatlán\...,/el-salvador-es/bienes-raices-venta-de-propied...,True,Antiguo Cuscatlán,28/08/2025,NaN,"https://www.google.com/maps/search/13.671771,+..."


In [ ]:

# https://www.google.com/maps/search/13.660035,+-89.262488
df.shape

(2250, 20)